# Metronome Compass Calibration

A Python port of RNGReporter's HGSS **Seed to Time** verification panel, built for
gathering Metronome-compass calibration data.

Give it a target datetime + delay and a search window, and it enumerates every candidate
seed nearby.  For each seed it reports:

- **Roamer relocation** — where Raikou / Entei / Latios(Latias) move to when the save is
  reloaded (given where they are now).
- **Elm phone-call sequence** — the `P`/`E`/`K` calls you can read off in-game.

Two sections identify the seed two ways: **Section A** from roamer routes + Elm calls
(`a_seed`), **Section B** from the Metronome battle (`b_seed`).  All logic lives in
`utils/calibration_tools.py`.

## Keyboard fixup

The `2` and `w` keys on my keyboard are flaky, so every `input()` prompt below accepts
`\T` for `2` and `\V` for `w` (substituted before the value is used).  It's applied
automatically at each interactive step — ipykernel resets `input` once per cell, so the
library re-installs the fixup at every prompt.  To add pairs, edit `INPUT_SUBS` in
`utils/calibration_tools.py`.


In [1]:
%load_ext autoreload
%autoreload 2
import datetime as dt
from utils.calibration_tools import (
    # Section A -- roamer routes + Elm calls
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section B -- Metronome-compass battle
    generate_candidates_near,
    print_candidates,
    narrow_candidates,
    prompt_magikarp,
    # Persist a run
    save_compass_run,
    # Timer calibration math
    calibrate_timer,
    # Review + apply a model update (deliberate; not automatic on save)
    update_calibration_model,
)

## Section A — Roamer + Elm identification  (→ `a_seed`)

**Configure** your target datetime/delay, the search window, and each roamer's **current**
route (where it is *now*, before the reset) plus whether it's still roaming.  Use `0` for a
current route you don't know or care about.  Variables are `a_`-prefixed so they won't clash
with Section B.

**Identify** (after loading the save, read the roamer map and Elm phone):

1. **Roamer routes** — one number per *roaming* legendary in **R E L** order (e.g. `38 42 11`);
   `.` leaves a roamer unconstrained.  Only roamers marked `present` are expected.
2. If more than one candidate matches, **Elm calls** — type `P`/`E`/`K` as you hear each
   call (matched as a substring, since RNG may advance first); other characters are ignored.
   Type `M` to pick a candidate by number instead.
3. The single surviving row is saved to **`a_seed`** (integer seed is `a_seed["seed"]`).


In [5]:
# --- Section A: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)
a_target_delay   = 681
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset).  A roamer roams iff it appears here.
a_prev_routes = {"r": 43, "e": 45, "l": 6}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, display_limit=a_display_limit)
# GOALS: EKP, KPEK, PEKK, EKKP


Observed roamer routes (R E L, space-separated, . = any):  33 39 10



Observed R=33 E=39 L=10  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02BC  2025-07-24 14:45:54     675    -6   -1   33  39  10   3  KKEEEEKEEEKKKKK
  0x0C0E02BC  2025-07-24 14:45:55     675    -6   +0   33  39  10   3  EPPPEKPEEKPKEPP
  0x0D0E02BC  2025-07-24 14:45:56     675    -6   +1   33  39  10   3  PEKKEPKKKPEEKEP


Elm calls (type P/E/K as heard; M = pick manually):  ppp


Elm calls so far: PPP
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02BC  2025-07-24 14:45:55     675    -6   +0   33  39  10   3  EPPPEKPEEKPKEPP

=== Seed identified: 0x0C0E02BC  2025-07-24 14:45:55  delay=675  R/E/L=33/39/10  Elm=EPPPEKPEEKPKEPP ===


## Section B — Expedition-style Seed Identification  (→ `b_seed`)

Ported from the Metronome Compass Testing notebook.  Generate every candidate seed near a
target `(time, delay)`, each with its precomputed Metronome battle path, then walk the real
battle turn by turn — candidates whose path diverges from what you observe drop out until a
single seed remains, saved as **`b_seed`** (the whole row).

Config uses `b_`-prefixed names so it won't clash with Section A.  The Metronome user's
movepool besides Metronome is `DEFAULT_EXTRA_MOVES` in `utils/calibration_tools.py`; edit it
there if it changes.  (Seeds here use the same year-correct `seed_for` as Section A.)

Magikarp's **level and gender are asked at run time** (they change each battle); the Metronome user's own gender is the stable `b_metronome_user_is_female` config.


In [6]:
# --- Section B: Metronome-compass target ---
# b_target_time     = dt.datetime(2025, 7, 24, 14, 49, 0)
b_timer_delay = 180055
b_target_time = a_target_time + dt.timedelta(milliseconds=b_timer_delay+5000)
b_target_delay    = 11202-25
b_seconds_window  = 2         # +/- X seconds
b_delay_window    = 2000       # +/- Y delays
b_metronome_only  = False     # True = Metronome-only user; False = + DEFAULT_EXTRA_MOVES
b_metronome_user_is_female = True   # the Metronome user's gender (rarely changes)

# Magikarp's level + gender change every run, so prompt for them at execution time.
# opposite_gender is derived relative to the Metronome user's gender above.
b_magikarp_level, b_opposite_gender = prompt_magikarp(b_metronome_user_is_female)

b_candidates = generate_candidates_near(
    b_target_time, b_target_delay, b_seconds_window, b_delay_window,
    magikarp_level=b_magikarp_level, opposite_gender=b_opposite_gender,
    metronome_only=b_metronome_only,
)

# Walk the real battle turn by turn; candidates diverging from what you observe drop out.
b_seed = narrow_candidates(b_candidates, b_magikarp_level, b_opposite_gender,
                           metronome_only=b_metronome_only)


Magikarp level:  16
Magikarp gender (M/F):  M



20005 / 20005 seeds remain -- next is turn 1
        Seed   Delay    dD  predicted turn 1
  0xD90E2BC2   11177    +0  KspM163h           (Slash)
  0x130E2BC2   11177    +0  KspM096            (Meditate)
  0xDA0E2BC2   11177    +0  KspM409h           (Drain Punch)
  0x120E2BC2   11177    +0  KspM268            (Charge)
  0xDB0E2BC2   11177    +0  KspM237h           (Hidden Power)
  0xD90E2BC1   11176    -1  KspM359!           (Hammer Arm)
  0xD90E2BC3   11178    +1  KtkhM449h          (Judgment)
  0x130E2BC1   11176    -1  KspM411h           (Focus Blast)
  0x130E2BC3   11178    +1  KtkhM105           (Recover)
  0xDA0E2BC1   11176    -1  KspM413h           (Brave Bird)
  0xDA0E2BC3   11178    +1  KtkhM124h          (Sludge)
  0x120E2BC1   11176    -1  KspM116            (Focus Energy)
  0x120E2BC3   11178    +1  KtkhM001h          (Pound)
  0xDB0E2BC1   11176    -1  KspM085h           (Thunderbolt)
  0xDB0E2BC3   11178    +1  KtkhM110           (Withdraw)
  ... and 19990 more

--- Tur

  Magikarp used? (sp/tk):  sp
  Metronome selected? (move name or M###):  Whirlpool
  Hit, crit, or miss? (h/!/-):  h



14 / 20005 seeds remain -- next is turn 2
        Seed   Delay    dD  predicted turn 2
  0xD90E2BC8   11183    +6  KspM441hBD         (Gunk Shot)
  0x130E2B93   11130   -47  KspM017hBD         (Wing Attack)
  0x120E2C56   11325  +148  KtkhM233hBD        (Vital Throw)
  0x120E2B0A   10993  -184  KtkhM065hBD        (Drill Peck)
  0x120E2D43   11562  +385  KspM046h_          (Roar)
  0xD90E2930   10519  -658  KspM022!BD         (Vine Whip)
  0xDB0E280F   10230  -947  KspM004-BD         (Comet Punch)
  0x120E2F83   12138  +961  KtkhM247h~BD       (Shadow Ball)
  0xDB0E27DA   10177  -1000  KtkhM295h~BD       (Luster Purge)
  0xDA0E3006   12269  +1092  KspM332hBD         (Aerial Ace)
  0xDB0E26ED    9940  -1237  KtkhM169BD         (Spider Web)
  0xDA0E30F3   12506  +1329  Ktk-M394hBD        (Flare Blitz)
  0xDB0E32F8   13023  +1846  KspM170BD          (Mind Reader)
  0xDB0E2455    9276  -1901  KtkhM145hBD        (Bubble)

--- Turn 2: answer what happened in the battle (or type ABORT to stop

  Magikarp used? (sp/tk):  sp
  Metronome selected? (move name or M###):  Gunk Shot
  Hit, crit, or miss? (h/!/-):  h
  Did the effect proc? (y/n):  n



1 / 20005 seeds remain -- next is turn 3
        Seed   Delay    dD  predicted turn 3
  0xD90E2BC8   11183    +6  KspM176BD          (Conversion 2)

Seed identified: 0xD90E2BC8  time=2025-07-24 14:49:00  delay=11183  dD=+6
Full path: KspM250hBD KspM441hBD KspM176BD KtkhM186hBF KtkhM152h CFZM272 SCFZKtkhM422h KtkhM159 KspM141h KspM042hhh
Remaining Metronome moves (turn 3+):
  Turn 3: Conversion 2 (M176)
  Turn 4: Sweet Kiss (M186)
  Turn 5: Crabhammer (M152)
  Turn 6: Role Play (M272)
  Turn 7: Thunder Fang (M422)
  Turn 8: Sharpen (M159)
  Turn 9: Leech Life (M141)
  Turn 10: Pin Missile (M042)


## Section C — Save the run  (→ `data/compass_runs.jsonl`)

Records this calibration run — both identified seeds (`a_seed`, `b_seed`) plus the metadata
below — as one JSON line appended to `data/compass_runs.jsonl`.

You're prompted for a **run tag** (e.g. `300s Samwise`, `11000d Work`), the **target timer
delay**, the **target timer calibration**, and free-form **notes**.  Leaving the tag / delay
/ calibration blank re-uses the previous run's value (notes never default).  The record is
pretty-printed and confirmed (`y`/`n`) before it's written.

> **Saving no longer touches the shared calibration model.** The chart/expedition read
> `data/calibration_model.json`, and changing it invalidates a precomputed chart (~1 h to
> rebuild) — you often save test runs *while* charting a target from the current model. Apply
> model changes deliberately in **Section E**.

In [7]:
# --- Section C: append this run to data/compass_runs.jsonl ---
# Saving does NOT change the shared calibration model (that would invalidate a precomputed
# chart).  Apply model changes deliberately in Section E below.
run_record = save_compass_run(a_seed, b_seed)

Run tag [Compass Target 4]:  Compass Target 5
Target timer delay [327496]:  180055
Target timer calibration [0]:  
Notes:  



{
  "saved_at": "2026-09-10T20:39:29",
  "tag": "Compass Target 5",
  "target_timer_delay": 180055,
  "target_timer_calibration": 0,
  "fresh_boot": true,
  "prior_battles": 0,
  "notes": "",
  "a_seed": {
    "seed": 202244796,
    "seed_hex": "0x0C0E02BC",
    "time": "2025-07-24T14:45:55",
    "delay": 675,
    "sec_delta": 0,
    "delay_delta": -6,
    "r_route": 33,
    "e_route": 39,
    "l_route": 10,
    "rng_calls": 3,
    "elm": "EPPPEKPEEKPKEPP"
  },
  "b_seed": {
    "seed": 3641584584,
    "seed_hex": "0xD90E2BC8",
    "time": "2025-07-24T14:49:00",
    "delay": 11183,
    "sec_delta": 0,
    "delay_delta": 6,
    "path_str": "KspM250hBD KspM441hBD KspM176BD KtkhM186hBF KtkhM152h CFZM272 SCFZKtkhM422h KtkhM159 KspM141h KspM042hhh"
  }
}



Save this run? (y/n):  y


Saved to data/compass_runs.jsonl
Calibration model NOT updated (run update_calibration_model() to review + apply changes).


## Section D — Timer calibration  (delay/calibration ↔ battle frame)

Fits the collected runs to answer: **given a timer countdown, which battle frame `F_b` will I hit?**
`M = target_timer_delay + target_timer_calibration` (ms, calibration signed).

**The model.** The deployed fit predicts `dF = F_b − F_a` against `M` and reconstructs the actual
battle-seed frame as `F_b = dF(M) + F_a`, where `F_a` is your initial seed's low16
(`a_seed & 0xFFFF`, which carries the year) — so the same fit works in any year. Two shapes are
fit and deployed together; the chart's `fps_model` picks one:

- **`linear`** (default) — a straight line; physically-sane slope, safe across the whole range.
- **`quad`** — adds curvature; fits 3–10 min slightly tighter, but its slope runs past the
  ~59.83 Hz hardware ceiling, so don't extrapolate beyond the calibrated range.

The `F_b`-direct and within-run-rate fits are also shown as diagnostic baselines — their RMS
sanity-checks the dF fit, and the within-run rate reads out the raw frames/sec.

**Uncertainty** splits two ways: reducible **mean-uncertainty** (shrinks with more runs; ~0 near
measured delays, grows on extrapolation) and irreducible **physical jitter** (`σ ≈ c·√M`) — the
latter sets your real hit odds, so pick a `tolerance` window covering the frames you would accept.


In [2]:
# --- Section D: fit the collected runs; predict the battle frame for a timer delay ---
model = calibrate_timer()   # reads data/compass_runs.jsonl, prints the fit report

# === Forward prediction: for a timer delay + calibration, which frame will I land on? ===
d_delay       = 327919    # target_timer_delay (ms)
d_calibration = 0         # target_timer_calibration (ms, signed)
d_Fa          = 706       # initial-seed low16 = a_seed & 0xFFFF (carries the year)

# The deployed model predicts dF; the actual battle frame is dF + F_a.
rec = model["models"][model["recommended"]]
pr  = rec["predict"](d_delay, d_calibration)
frame = pr["expected"] + d_Fa
print(f"\nM = {d_delay + d_calibration} ms  ->  dF = {pr['expected']:.1f},  "
      f"actual F_b = dF + F_a({d_Fa}) = {frame:.1f}   (jitter +/-{pr['jitter']:.1f})")

# === How likely to land on a specific frame (within a +/- window)? ===
d_target_frame = round(frame)   # default: the predicted frame
d_tolerance    = 25             # half-window (frames) counted as a hit
hp = rec["hit_probability"](d_delay, d_calibration, d_target_frame - d_Fa, tolerance=d_tolerance)
print(f"P(land on F_b={d_target_frame} +/-{d_tolerance}) = {hp['p']*100:.1f}%  "
      f"(off center by {hp['delta']:+.1f} frames)")

# === Reverse: which delay centers you on a target frame? ===
sol = rec["solve"](d_target_frame - d_Fa, calibration=d_calibration)
print(f"to center on F_b={d_target_frame} (cal={d_calibration}): delay = {sol['delay']:.0f} ms")


=== Timer calibration  (42 run(s), 42 timed, 1 excluded as outlier) ===

  F_b as a function of M = delay + calibration (ms)

  within-run avg rate 58.0688 +/- 0.0117 Hz (dF/dt; rises with M as the slow post-boot frames dilute out)

   within-run-rate slope                    [Fb] slope 58.0688 Hz                 RMS residual  275.8 frames
   F_b line                                 [Fb] slope 59.9903 Hz                 RMS residual   70.7 frames
   F_b quadratic                            [Fb] inst rate 59.550->60.970 Hz over M range RMS residual   57.3 frames
  *dF line (deployed)                       [dF] slope 59.9445 Hz                 RMS residual   74.6 frames
   dF quadratic (deployed, fps_model=quad)  [dF] inst rate 59.514->61.033 Hz over M range RMS residual   57.2 frames

  ( * = deployed default; predict/solve/hit_probability use it. [dF] models reconstruct F_b = dF + F_a )

  RTC-second offset 5.28 +/- 0.52 s  (battle RTC second = round(M/1000 + this); +/-1 s is timestamp

## Section E — Apply the model update  (→ `data/calibration_model.json`)

Section D only *reports* the fit; it doesn't change anything. This cell re-fits from all runs,
shows how each parameter differs from the **currently deployed** `data/calibration_model.json`,
and writes the new model **only after you confirm**.

Run it when you actually want the chart/expedition to adopt the latest calibration — **not**
every time you save a test run. Because the chart reads this artifact, applying a change means
the next `x.precompute_chart()` will rebuild (~1 h), so do it deliberately between charting
sessions.

In [4]:
# --- Section E: review the re-fit vs the deployed model, then write it only if confirmed ---
# Prints an old -> new parameter table and asks before overwriting data/calibration_model.json.
# Applying it means the chart is stale until you re-run x.precompute_chart().
new_model = update_calibration_model()


[linear]
Proposed calibration-model change (old -> new):
  kind                           line -> line          
  n_runs                           42 -> 42            
  n_fit                            41 -> 41            
  beta                       0.059944 -> 0.059944      
  alpha                       -297.29 -> -297.29       
  coeffs                           () -> ()            
  jitter_c                    0.11983 -> 0.11983       
  jitter_rms                   74.643 -> 74.643        
  rtc_offset_seconds           5.2831 -> 5.2831        
  rtc_offset_std               0.5174 -> 0.5174        
  m_lo                         175000 -> 175000        
  m_hi                         595000 -> 595000        

[quad]
Proposed calibration-model change (old -> new):
  kind                           quad -> quad          
  n_runs                           42 -> 42            
  n_fit                            41 -> 41            
  beta                              0 -> 0    